In [1]:
'''IMPORTS'''
import numpy as np
import os
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import sys

sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))
from data.data_utils import *

sys.path.append(os.path.abspath('..'))
import custom_utils

In [2]:
'''SETTINGS'''
data_path = "/home/crw1213/ece477-final-project/data"
seed = 42

In [3]:
'''WRAPPER FUNCTIONS'''
def preprocess(base_data_path, dataset_name, has_cat_features, 
               has_num_features, real_data=True, syn_type=None):
    # Load data
    if real_data:
        X_num_train, X_num_test, X_num_val, y_train, y_test, y_val = custom_utils.load_data(base_data_path=base_data_path,
                                                                                            dataset_name=dataset_name,
                                                                                            has_cat_features=has_cat_features,
                                                                                            has_num_features=has_num_features)
    else: 
        X_num_train, X_num_test, X_num_val, y_train, y_test, y_val = custom_utils.load_syn_data(base_data_path=base_data_path,
                                                                                                dataset_name=dataset_name,
                                                                                                syn_type=syn_type,
                                                                                                has_cat_features=has_cat_features,
                                                                                                has_num_features=has_num_features)
    

    # Clip outliers in each feature column for continuous values
    for i in range(X_num_train.shape[1]):
        lower = np.percentile(X_num_train[:, i], 0.5)
        upper = np.percentile(X_num_train[:, i], 99.5)

        X_num_train[X_num_train[:, i] < lower, i] = lower 
        X_num_train[X_num_train[:, i] > upper, i] = upper

        X_num_test[X_num_test[:, i] < lower, i] = lower 
        X_num_test[X_num_test[:, i] > upper, i] = upper

        X_num_val[X_num_val[:, i] < lower, i] = lower 
        X_num_val[X_num_val[:, i] > upper, i] = upper

    # Convert back into DataFrames for compatbility with DLN package
    num_features = ['radius1', 'texture1', 'perimeter1', 'area1', 'smoothness1',
                    'compactness1', 'concavity1', 'concave_points1', 'symmetry1',
                    'fractal_dimension1', 'radius2', 'texture2', 'perimeter2', 'area2',
                    'smoothness2', 'compactness2', 'concavity2', 'concave_points2',
                    'symmetry2', 'fractal_dimension2', 'radius3', 'texture3', 'perimeter3',
                    'area3', 'smoothness3', 'compactness3', 'concavity3', 'concave_points3',
                    'symmetry3', 'fractal_dimension3']

    X_num_train_df = pd.DataFrame(X_num_train, columns=num_features)
    X_num_test_df = pd.DataFrame(X_num_test, columns=num_features)
    X_num_val_df = pd.DataFrame(X_num_val, columns=num_features)

    cat_features = []

    y_train_df = pd.DataFrame(y_train, columns=["Target"])
    y_test_df = pd.DataFrame(y_test, columns=["Target"])
    y_val_df = pd.DataFrame(y_val, columns=["Target"])

    train_df = pd.concat([X_num_train_df, y_train_df], axis=1)
    test_df = pd.concat([X_num_test_df, y_test_df], axis=1)
    val_df = pd.concat([X_num_val_df, y_val_df], axis=1)

    # Normalization
    scaler_list = [MinMaxScaler(clip=True), MinMaxScaler(clip=True)]
    feature_list = [num_features, cat_features]
    train_scaled_df, test_scaled_df, val_scaled_df, scaler_params = custom_utils.scale_features(df_train=train_df,
                                                                                                df_test=test_df,
                                                                                                df_val=val_df,
                                                                                                cols_list=feature_list,
                                                                                                scaler_list=scaler_list)
    dtype_dict = train_scaled_df.dtypes.to_dict()

    # Save
    if syn_type is not None:
        folderpath = f'/home/crw1213/ece477-final-project/evals/dln/data/datasets/breast_cancer_{syn_type}/seed_{seed}/data'
    else:
        folderpath = f'/home/crw1213/ece477-final-project/evals/dln/data/datasets/breast_cancer/seed_{seed}/data'
        
    save_data(folderpath, num_features, cat_features, scaler_params, dtype_dict, train_scaled_df, test_scaled_df, val_scaled_df)

BREAST CANCER

In [4]:
# Real data
preprocess(base_data_path=data_path, dataset_name="breast_cancer", has_cat_features=False, 
           has_num_features=True, real_data=True, syn_type=None)

In [5]:
# KDE_Lab data
preprocess(base_data_path=data_path, dataset_name="breast_cancer", has_cat_features=False, 
           has_num_features=True, real_data=False, syn_type="kde_lab")

In [6]:
# GMME_Lab data
preprocess(base_data_path=data_path, dataset_name="breast_cancer", has_cat_features=False, 
           has_num_features=True, real_data=False, syn_type="gmme_lab")

In [7]:
# DDPM data
preprocess(base_data_path=data_path, dataset_name="breast_cancer", has_cat_features=False, 
           has_num_features=True, real_data=False, syn_type="ddpm")

In [9]:
# SMOTE data
preprocess(base_data_path=data_path, dataset_name="breast_cancer", has_cat_features=False, 
           has_num_features=True, real_data=False, syn_type="smote")